# 28.1 — Triplet Encoder + ANCE Hard Negative Mining (WJ 512)

Same encoder as nb28 but replaces **in-batch hard negatives** with **ANCE-style global hard negative mining**:
every `mine_every` epochs, encode the full corpus, build a temporary ANN index, and pick the
hardest non-GT corpus neighbor per query as an explicit negative. No B×B cross-similarity matrix —
each step is a clean (query, positive, hard_negative) triplet.

In [ ]:
import os, random, sys, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
sys.path.append('/raid/ruban/hpmlproj/term_project')
from sota_experiment_common import (
    cleanup, eval_recall, l1_simplex, load_dataset, load_dataset_normalized,
    nmslib_neighbors, preload_rerank_corpus, release_rerank_corpus, rerank_wj_gpu, save_result,
)

dataset_name = "full"
out_dim      = 512
device       = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
THREADS      = 40
seed         = 42
batch_size   = 2048
epochs       = 75
lr           = 1e-3
weight_decay = 1e-4
max_pos      = 30
margin       = 0.5   # increased from 0.3 — forces model to push negatives further, keeps gradient alive
mine_every   = 2
mine_k       = 2000
candidate_ks = [500, 1000] if dataset_name == "10k" else [1000, 2000]

# Early stopping
es_viol_threshold = 5.0   # avg violations/step below this = saturation
es_viol_strikes   = 3     # consecutive re-mine epochs with low viol before stopping
es_loss_patience  = 15    # epochs without loss improvement before stopping
es_min_delta      = 1e-4  # minimum improvement to count as progress

METHOD_NAME   = "triplet_ance_wj_512"
NOTEBOOK_NAME = "28_1_triplet_ance_wj_512.ipynb"
OUT_PATH      = "/tmp/results_sota_triplet_ance_wj_512.pkl"
CKPT_PATH     = "/tmp/best_sota_triplet_ance_wj_512_full.pt"

random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
print(f"device={device} | batch={batch_size} | epochs={epochs} | margin={margin} | mine_every={mine_every} | mine_k={mine_k}")

In [6]:
qt, gt, query_start, corpus_qt, query_qt, corpus_sums, qt_norm = load_dataset_normalized(dataset_name)

dataset=full | qt=(233773, 18220) | corpus=(187019, 18220) | queries=(46754, 18220)
Computing qt_norm (first run — caching to /tmp)...
qt_norm computed+cached (233773, 18220) (15.87 GB) in 415.1s


In [7]:
def wj_sim(a, b):
    mins = torch.minimum(a, b).sum(dim=-1)
    maxs = torch.maximum(a, b).sum(dim=-1).clamp(min=1e-10)
    return mins / maxs

def ance_triplet_loss(zq, zp, zn, margin=0.3):
    """Explicit triplet loss on pre-mined (query, positive, hard_negative) triples.
    No B×B matrix — scales to any corpus size."""
    sim_pos = wj_sim(zq, zp)
    sim_neg = wj_sim(zq, zn)
    loss    = F.relu(sim_neg - sim_pos + margin)
    violated = loss > 0
    if violated.sum() == 0:
        return torch.tensor(0.0, device=zq.device, requires_grad=True), 0
    return loss[violated].mean(), int(violated.sum().item())

class ANCEDataset(Dataset):
    """Yields (query_id, positive_id, hard_neg_id) triples using pre-mined hard negatives."""
    def __init__(self, gt_lookup, query_start, hard_negs, max_pos=30):
        self.pairs = []
        for qid, neighbors in gt_lookup.items():
            if qid < query_start or qid not in hard_negs:
                continue
            neg_id = hard_negs[qid]
            for nid in neighbors[:max_pos]:
                if nid < query_start:
                    self.pairs.append((qid, nid, neg_id))
        random.shuffle(self.pairs)
        n_q = len(set(p[0] for p in self.pairs))
        print(f"  triplets={len(self.pairs):,} | queries_with_neg={n_q} | steps/epoch={len(self.pairs)//batch_size}")
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx): return self.pairs[idx]

def embed_all(model, qt, batch_size=4096):
    """Encode using DataParallel across all GPUs. 4096 batch = 512/GPU on 8 GPUs."""
    model.eval(); out = []
    with torch.no_grad():
        for s in range(0, len(qt), batch_size):
            x = torch.tensor(qt[s:s+batch_size], dtype=torch.float32, device=device)
            out.append(model(x).cpu().numpy().astype(np.float32))
    return np.vstack(out)

def mine_hard_negatives(model, qt_norm, gt, query_start, k=200, threads=THREADS):
    """
    Encode all corpus+query items, build temp ANN, find hardest non-GT corpus
    neighbor per query. Returns dict: global_query_id -> global_corpus_id.
    """
    print("  Mining...", end=" ", flush=True)
    t0   = time.time()
    embs = embed_all(model, qt_norm)          # all 8 GPUs, batch=4096
    corpus_embs = embs[:query_start]
    query_embs  = embs[query_start:]
    nbrs, _ = nmslib_neighbors(corpus_embs, query_embs,
                                space="WeightedJaccard", k=k, threads=threads)
    hard_negs = {}
    for q_offset in range(len(query_embs)):
        q_id   = query_start + q_offset
        gt_set = set(gt.get(q_id, []))
        for cand in nbrs[q_offset]:
            if int(cand) >= 0 and int(cand) not in gt_set:
                hard_negs[q_id] = int(cand)
                break
    print(f"found {len(hard_negs)}/{len(query_embs)} hard negs in {time.time()-t0:.1f}s", flush=True)
    return hard_negs

class TripletEncoder(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def encode(self, x):
        z = F.relu(self.encoder(x))
        return z / z.sum(dim=1, keepdim=True).clamp(min=1e-10)
    def forward(self, x):
        return self.encode(x)

def eval_embeddings(embs, method_name, out_path, notebook_name):
    corpus_embs = embs[:query_start]; query_embs = embs[query_start:]
    max_k = max(max(candidate_ks), 500)
    nbrs, info = nmslib_neighbors(corpus_embs, query_embs, space="WeightedJaccard", k=max_k, threads=THREADS)
    metrics = {**eval_recall(gt, nbrs, query_start, max_k), **info, "dim": out_dim}
    for k, v in metrics.items():
        if isinstance(k, int): print(f"R@{k:<4} = {v:.4f}")
    print(f"QPS={metrics['qps']:.1f}")
    save_result(out_path, dataset_name, method_name, metrics, meta={"notebook": notebook_name})
    preload_rerank_corpus(corpus_qt, corpus_sums)
    for ck in candidate_ks:
        cand, ci = nmslib_neighbors(corpus_embs, query_embs, space="WeightedJaccard", k=ck, threads=THREADS)
        t0 = time.time()
        rr = rerank_wj_gpu(query_qt, cand, corpus_qt, corpus_sums, top_k=ck, batch_size=8)
        qps_total = len(query_qt) / max(time.time()-t0 + len(query_qt)/max(ci['qps'],1e-9), 1e-9)
        rr_metrics = {**eval_recall(gt, rr, query_start, ck), "qps": qps_total, "candidate_k": ck}
        key = f"{method_name}_rerank_{ck}"
        for k, v in rr_metrics.items():
            if isinstance(k, int): print(f"{key} R@{k} = {v:.4f}")
        print(f"{key} QPS={rr_metrics['qps']:.1f}")
        save_result(out_path, dataset_name, key, rr_metrics, meta={"notebook": notebook_name})
    release_rerank_corpus()

In [8]:
device      = torch.device("cuda:0")
vecs_device = torch.device("cuda:7")
print("Pre-loading vectors to cuda:7...")
vecs_gpu = torch.from_numpy(np.ascontiguousarray(qt_norm, dtype=np.float32)).to(vecs_device)
print(f"Loaded: {vecs_gpu.nbytes/1024**3:.2f} GB on {vecs_device}")

model = TripletEncoder(qt_norm.shape[1], out_dim)
model = nn.DataParallel(model, device_ids=list(range(torch.cuda.device_count())))
model = model.to(device)
print(f"DataParallel on {torch.cuda.device_count()} GPUs")

opt  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
best = float('inf')
t0_train = time.time()

def make_loader(hard_negs):
    ds = ANCEDataset(gt, query_start, hard_negs, max_pos=max_pos)
    return DataLoader(ds, batch_size=batch_size, shuffle=True,
                      num_workers=4, pin_memory=True, drop_last=True)

# Initial hard negative mine before epoch 1
print(f"\n[init] Hard negative mining (k={mine_k}):")
hard_negs = mine_hard_negatives(model, qt_norm, gt, query_start, k=mine_k)
loader    = make_loader(hard_negs)

epoch_bar = tqdm(range(1, epochs + 1), desc="epochs", unit="ep")
for epoch in epoch_bar:
    # Re-mine every mine_every epochs (epochs 6, 11, 16, ...)
    if epoch > 1 and (epoch - 1) % mine_every == 0:
        print(f"\n[ep{epoch:02d}] Re-mining (k={mine_k}):", flush=True)
        hard_negs = mine_hard_negatives(model, qt_norm, gt, query_start, k=mine_k)
        loader    = make_loader(hard_negs)

    model.train()
    tot_loss = tot_viol = steps = 0
    step_bar = tqdm(loader, desc=f"ep{epoch:02d}", leave=False, unit="step")
    for q_ids, p_ids, n_ids in step_bar:
        q = vecs_gpu[q_ids.to(vecs_device)].to(device)
        p = vecs_gpu[p_ids.to(vecs_device)].to(device)
        n = vecs_gpu[n_ids.to(vecs_device)].to(device)
        B = q.shape[0]
        z          = model(torch.cat([q, p, n]))
        zq, zp, zn = z[:B], z[B:2*B], z[2*B:]
        loss, n_viol = ance_triplet_loss(zq, zp, zn, margin=margin)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        tot_loss += float(loss.detach()); tot_viol += n_viol; steps += 1
        step_bar.set_postfix(loss=f"{float(loss.detach()):.4f}", viol=n_viol)
    sch.step()
    avg = tot_loss / max(steps, 1)
    if avg < best:
        best = avg
        torch.save(model.module.state_dict(), CKPT_PATH)
    elapsed = (time.time() - t0_train) / 60
    eta     = elapsed / epoch * (epochs - epoch)
    epoch_bar.set_postfix(loss=f"{avg:.4f}", best=f"{best:.4f}", eta=f"{eta:.0f}m")
    if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
        print(f"epoch {epoch:02d}/{epochs} | loss={avg:.4f} | viol={tot_viol/steps:.1f} | "
              f"{elapsed:.1f}min | eta={eta:.1f}min", flush=True)

print(f"\nTraining done. best={best:.4f} | saved {CKPT_PATH}")

Pre-loading vectors to cuda:7...
Loaded: 15.87 GB on cuda:7
DataParallel on 8 GPUs

[init] Hard negative mining (k=200):
  Mining... 

/raid/ruban/installs/miniconda3/envs/hpmlproj/lib/python3.10/site-packages/torch/nn/modules/linear.py:125: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at ../aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return F.linear(input, self.weight, self.bias)

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
********************************************************

found 23660/46754 hard negs in 180.5s
  triplets=592,659 | queries_with_neg=21572 | steps/epoch=289


epochs:   0%|          | 0/75 [00:00<?, ?ep/s]

ep01:   0%|          | 0/289 [00:01<?, ?step/s]

...

ep72:   0%|          | 0/298 [00:00<?, ?step/s]

ep73:   0%|          | 0/298 [00:00<?, ?step/s]

ep74:   0%|          | 0/298 [00:00<?, ?step/s]

ep75:   0%|          | 0/298 [00:00<?, ?step/s]

epoch 75/75 | loss=0.0194 | viol=36.9 | 74.1min | eta=0.0min

Training done. best=0.0102 | saved /tmp/best_sota_triplet_ance_wj_512_full.pt


In [9]:
(model.module if hasattr(model, "module") else model).load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
embs = embed_all(model, qt_norm)
eval_embeddings(embs, METHOD_NAME, OUT_PATH, NOTEBOOK_NAME)
cleanup()



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

R@10   = 0.3543
R@50   = 0.4065
R@100  = 0.4326
R@500  = 0.5204
QPS=2040.8
saved triplet_ance_wj_512 -> /tmp/results_sota_triplet_ance_wj_512.pkl
Corpus pre-loaded to GPU: 12.69 GB



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

triplet_ance_wj_512_rerank_1000 R@10 = 0.9369
triplet_ance_wj_512_rerank_1000 R@50 = 0.8724
triplet_ance_wj_512_rerank_1000 R@100 = 0.8201
triplet_ance_wj_512_rerank_1000 R@500 = 0.6439
triplet_ance_wj_512_rerank_1000 QPS=938.5
saved triplet_ance_wj_512_rerank_1000 -> /tmp/results_sota_triplet_ance_wj_512.pkl



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

triplet_ance_wj_512_rerank_2000 R@10 = 0.9463
triplet_ance_wj_512_rerank_2000 R@50 = 0.8876
triplet_ance_wj_512_rerank_2000 R@100 = 0.8380
triplet_ance_wj_512_rerank_2000 R@500 = 0.6654
triplet_ance_wj_512_rerank_2000 QPS=640.2
saved triplet_ance_wj_512_rerank_2000 -> /tmp/results_sota_triplet_ance_wj_512.pkl
